In [1]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse

app = FastAPI()


In [2]:
import whisper

asr_model = whisper.load_model("small")

def transcribe_audio(audio_bytes):
    with open("temp.wav", "wb") as f:
        f.write(audio_bytes)
    result = asr_model.transcribe("temp.wav")
    return result["text"]


In [3]:
import json
import sympy
import requests

def calculate(expression: str) -> str:
    try:
        result = sympy.sympify(expression)
        return str(result)
    except Exception as e:
        return f"Error in calculation: {str(e)}"

def search_arxiv(query: str) -> str:
    url = f"http://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results=1"
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        return "Error fetching data from arXiv."

AVAILABLE_FUNCTIONS = {
    "calculate": calculate,
    "search_arxiv": search_arxiv
}

FUNCTION_SCHEMAS = [
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression and return the result",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to evaluate, e.g., '2+2', 'sqrt(16)', 'sin(pi/2)'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "search_arxiv",
        "description": "Search arXiv for academic papers matching a query",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query for finding papers"
                }
            },
            "required": ["query"]
        }
    }
]

In [4]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

conversation_history = [
    {
        "role": "system",
        "content": """You are a helpful voice assistant with access to special tools. 
When users ask questions that require:
- Mathematical calculations (arithmetic, algebra, calculus, etc.) → use the calculate function
- Searching for academic papers or research → use the search_arxiv function

Use these tools whenever appropriate to provide accurate answers. 
For general conversation, respond naturally without using tools.
Keep your responses concise and conversational since they will be spoken aloud."""
    }
]

def generate_response(user_text):
    # Add user message to history
    conversation_history.append({"role": "user", "content": user_text})
    
    # Call OpenAI API with function definitions
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=conversation_history,
        tools=[{"type": "function", "function": func} for func in FUNCTION_SCHEMAS],
        tool_choice="auto",  # Let model decide when to use functions
        max_tokens=150
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    if tool_calls:
        # Add assistant's tool call message to history (important: add the full message object)
        conversation_history.append(response_message.model_dump())
        
        # Execute each function call
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"Function call: {function_name}({function_args})")
            
            # Call the actual function
            function_response = AVAILABLE_FUNCTIONS[function_name](**function_args)
            
            print(f"Function result: {function_response}")
            
            # Add function result to conversation
            conversation_history.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response
            })
        
        # Get final response from model with function results
        second_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=conversation_history,
            max_tokens=150
        )
        
        bot_response = second_response.choices[0].message.content
    else:
        # No function call, use direct response
        bot_response = response_message.content
    
    # Add final assistant response to history
    conversation_history.append({"role": "assistant", "content": bot_response})
    
    return bot_response


In [5]:
# COSYVOICE TTS (Cosyvoice does not work properly both in this agent and in demo, potentially an issue with Windows, will try WSL in the future.)
# import sys
# import os
# cosyvoice_path = r'C:\Users\Kevin\Desktop\ai_class\KevinYuan_Homework\CosyVoice'
# sys.path.append(cosyvoice_path)
# sys.path.append(os.path.join(cosyvoice_path, 'third_party', 'Matcha-TTS'))
# from cosyvoice.cli.cosyvoice import CosyVoice2
# from cosyvoice.utils.file_utils import load_wav
# import torchaudio
# model_path = os.path.join(cosyvoice_path, 'pretrained_models', 'CosyVoice2-0.5B')
# cosyvoice = CosyVoice2(model_path, load_jit=False, load_trt=False, load_vllm=False, fp16=False)
# reference_audio_path = os.path.join(cosyvoice_path, 'asset', 'ENG_US_M_DAVEL.wav')
# # Transcription of the reference audio
# prompt_text = "Being able to communicate positions within a room is critical to our ability to focus light on a certain area and place objects in their proper location on stage. But it goes even deeper than that, this proficiency provides the basic vocabulary in a common language that is spoken by production and staging professionals around the world."
# prompt_speech_16k = load_wav(reference_audio_path, 16000)
# def synthesize_speech(text, output_path="response.wav"):
#     # Add text_frontend=False for better English synthesis (per CosyVoice docs)
#     for i, result in enumerate(cosyvoice.inference_zero_shot(
#         text,              # Text to synthesize
#         prompt_text,       # Transcript of reference audio
#         prompt_speech_16k, # Reference audio
#         stream=False,
#         text_frontend=False  # Important for English text
#     )):
#         torchaudio.save(output_path, result['tts_speech'], cosyvoice.sample_rate)
#         break
#     return output_path

def synthesize_speech(text, output_path="response.wav"):
    response = client.audio.speech.create(
        model="tts-1",
        voice="alloy", 
        input=text
    )
    response.stream_to_file(output_path)
    return output_path



In [6]:
@app.post("/chat/")
async def chat_endpoint(file: UploadFile = File(...)):
    audio_bytes = await file.read()
    user_text = transcribe_audio(audio_bytes)
    bot_text = generate_response(user_text)
    audio_path = synthesize_speech(bot_text)
    return FileResponse(audio_path, media_type="audio/wav")

In [7]:
import gradio as gr

def voice_chat(audio):
    if audio is None:
        return None, ""
    
    with open(audio, "rb") as f:
        audio_bytes = f.read()
    
    user_text = transcribe_audio(audio_bytes)
    print(f"User said: {user_text}")
    
    bot_text = generate_response(user_text)
    print(f"Bot responds: {bot_text}")
    
    audio_path = synthesize_speech(bot_text, "gradio_response.wav")
    
    return audio_path, bot_text

gr.Interface(
    fn=voice_chat,
    inputs=gr.Audio(sources=["microphone"], type="filepath"),
    outputs=[
        gr.Audio(label="Bot Voice Response"),
        gr.Textbox(label="Bot Text Response", lines=5)
    ]
).launch()


c:\Users\Kevin\Desktop\ai_class\KevinYuan_Homework\week6_HW\venv_py312_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\Kevin\Desktop\ai_class\KevinYuan_Homework\week6_HW\venv_py312_gpu\Lib\site-packages\whisper\model.py:124: UserWarning: 1Torch was not compiled with memory efficient attention. (Triggered internally at C:\develop\pytorch-test\aten\src\ATen\native\transformers\hip\sdp_utils.cpp:726.)
  a = scaled_dot_product_attention(


User said:  What is 2 to the power of 32?
Function call: calculate({'expression': '2**32'})
Function result: 4294967296
Bot responds: Two to the power of 32 is 4,294,967,296.


C:\Users\Kevin\AppData\Local\Temp\ipykernel_55256\69168491.py:35: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_path)


User said:  Find papers on natural language processing.
Function call: search_arxiv({'query': 'natural language processing'})
Function result: <?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/+lm6Tka4UAQJ+8Xgvqenr0MTjuQ</id>
  <title>arXiv Query: search_query=all:natural OR all:language OR all:processing&amp;id_list=&amp;start=0&amp;max_results=1</title>
  <updated>2026-01-15T16:50:27Z</updated>
  <link href="https://arxiv.org/api/query?search_query=all:natural+OR+(all:language+OR+all:processing)&amp;start=0&amp;max_results=1&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>1</opensearch:itemsPerPage>
  <opensearch:totalResults>766547</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2110.10780v3</id>
    <title>An Open Natural Language Proc

C:\Users\Kevin\AppData\Local\Temp\ipykernel_55256\69168491.py:35: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_path)


User said:  Can you name me the best teams in the NBA?
Bot responds: Some of the best teams in the NBA historically include the Los Angeles Lakers, Boston Celtics, Chicago Bulls, Golden State Warriors, and San Antonio Spurs. Recently, teams like the Milwaukee Bucks and Miami Heat have also been very competitive.


C:\Users\Kevin\AppData\Local\Temp\ipykernel_55256\69168491.py:35: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_path)
